# 01 — Descarga de Datos Externos

Este notebook descarga los dos datasets que no están en Inside Airbnb:
- **Open-Meteo**: datos climáticos diarios de Barcelona
- **Ticketmaster**: eventos celebrados en Barcelona

Los ficheros resultantes se guardan en `data/raw/`.

> Nota: el rango de fechas está alineado con el `calendar.csv` de Inside Airbnb (scrape 14/12/2025): **2025-12-01 → 2026-12-31**.

## 0. Imports y configuración

In [1]:
import requests
import pandas as pd
import os
import time
import datetime as dt
from dotenv import load_dotenv
from pathlib import Path

# Carga las variables del fichero .env (TICKETMASTER_API_KEY, etc.)
load_dotenv()

# Rutas
RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Rutas OK')
print(f'Directorio de datos: {RAW_DIR.resolve()}')

Rutas OK
Directorio de datos: C:\Users\David\Documents\price-optimazer-guiris\data\raw


---
## 1. Open-Meteo — Clima de Barcelona

Descargamos datos diarios alineados con el rango del `calendar.csv` de Inside Airbnb (**2025-12-01 → 2026-12-31**).

La `archive-api` de Open-Meteo solo contiene histórico real (hasta ~5 días antes de hoy). Para las fechas futuras usamos el **mismo rango del año anterior** como proxy climatológico y después desplazamos las fechas +1 año. Guardamos la columna `fuente` para distinguir filas observadas de climatológicas.

- API pública, sin key necesaria · Licencia: CC BY 4.0 · Docs: https://open-meteo.com/en/docs

In [2]:
OPENMETEO_URL = 'https://archive-api.open-meteo.com/v1/archive'

TARGET_START = dt.date(2025, 12, 1)
TARGET_END   = dt.date(2026, 12, 31)

# archive-api tiene un delay de ~5 días, cortamos el tramo "observado" ahí
today  = dt.date.today()
cutoff = today - dt.timedelta(days=6)
if cutoff < TARGET_START:
    cutoff = TARGET_START - dt.timedelta(days=1)   # todo el rango es futuro

DAILY_VARS = [
    'temperature_2m_max', 'temperature_2m_min',
    'precipitation_sum', 'windspeed_10m_max', 'weathercode'
]

def fetch_meteo(start: dt.date, end: dt.date) -> pd.DataFrame:
    """Descarga un tramo del archive-api y devuelve un DataFrame."""
    r = requests.get(OPENMETEO_URL, params={
        'latitude': 41.3828, 'longitude': 2.1769,
        'start_date': start.isoformat(), 'end_date': end.isoformat(),
        'daily': DAILY_VARS, 'timezone': 'Europe/Madrid'
    })
    r.raise_for_status()
    df = pd.DataFrame(r.json()['daily'])
    df.rename(columns={'time': 'date'}, inplace=True)
    df['date'] = pd.to_datetime(df['date'])
    return df

partes = []

# --- Tramo 1: pasado real (observado) ---
if cutoff >= TARGET_START:
    print(f'Tramo observado: {TARGET_START} → {cutoff}')
    df_obs = fetch_meteo(TARGET_START, cutoff)
    df_obs['fuente'] = 'observado'
    partes.append(df_obs)
    print(f'  {len(df_obs)} días descargados')

# --- Tramo 2: futuro como climatología (mismo rango año anterior, luego shift +1 año) ---
future_start = max(cutoff + dt.timedelta(days=1), TARGET_START)
if future_start <= TARGET_END:
    clim_start = future_start.replace(year=future_start.year - 1)
    clim_end   = TARGET_END.replace(year=TARGET_END.year - 1)
    print(f'Tramo climatológico (año anterior): {clim_start} → {clim_end}')
    df_clim = fetch_meteo(clim_start, clim_end)
    df_clim['date'] = df_clim['date'] + pd.DateOffset(years=1)
    df_clim['fuente'] = 'climatologia'
    partes.append(df_clim)
    print(f'  {len(df_clim)} días descargados y desplazados +1 año')

df_clima = pd.concat(partes, ignore_index=True).sort_values('date').reset_index(drop=True)
print(f'\nTotal días: {len(df_clima)} ({df_clima["date"].min().date()} → {df_clima["date"].max().date()})')
print(df_clima['fuente'].value_counts())

Tramo observado: 2025-12-01 → 2026-04-16
  137 días descargados
Tramo climatológico (año anterior): 2025-04-17 → 2025-12-31
  259 días descargados y desplazados +1 año

Total días: 396 (2025-12-01 → 2026-12-31)
fuente
climatologia    259
observado       137
Name: count, dtype: int64


In [3]:
# Guardar
nulos = df_clima.isna().sum().sum()
if nulos:
    print(f'Aviso: {nulos} nulos en el dataframe')

output_path = RAW_DIR / 'clima_barcelona.csv'
df_clima.to_csv(output_path, index=False)

print(f'Guardado en: {output_path}')
print(f'Shape: {df_clima.shape}')
df_clima.head()

Guardado en: ..\data\raw\clima_barcelona.csv
Shape: (396, 7)


,date,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode,fuente
0,2025-12-01,15.3,5.0,0.0,10.0,3,observado
1,2025-12-02,15.7,5.9,0.0,12.9,3,observado
2,2025-12-03,14.6,6.0,0.4,14.3,51,observado
3,2025-12-04,10.4,5.0,1.6,11.9,53,observado
4,2025-12-05,15.6,6.3,0.0,12.4,3,observado


---
## 2. Ticketmaster — Eventos en Barcelona

Descargamos todos los eventos disponibles en Barcelona usando la **Discovery API v2**.

- Requiere Consumer Key (guardada en `.env` como `TICKETMASTER_API_KEY`)
- Licencia: uso no comercial permitido
- Docs: https://developer.ticketmaster.com/products-and-docs/apis/discovery-api/v2/

> La API devuelve máximo 200 eventos por página y limita a 5 páginas en el plan gratuito.

In [4]:
API_KEY = os.getenv('TICKETMASTER_API_KEY')

if not API_KEY:
    raise ValueError('No se encontró TICKETMASTER_API_KEY en el fichero .env')

print(f'API Key cargada: {API_KEY[:6]}...{API_KEY[-4:]}')

API Key cargada: 9ZL2vr...7let


In [5]:
TM_URL = 'https://app.ticketmaster.com/discovery/v2/events.json'

def fetch_events_page(page, start_date, end_date):
    """Descarga una página de eventos de Barcelona."""
    params = {
        'apikey': API_KEY,
        'city': 'Barcelona',
        'countryCode': 'ES',
        'startDateTime': f'{start_date}T00:00:00Z',
        'endDateTime': f'{end_date}T23:59:59Z',
        'size': 200,
        'page': page,
        'sort': 'date,asc'
    }
    response = requests.get(TM_URL, params=params)
    response.raise_for_status()
    return response.json()

def parse_events(data):
    """Extrae los campos relevantes de la respuesta de la API."""
    events = []
    for e in data.get('_embedded', {}).get('events', []):
        events.append({
            'id': e.get('id'),
            'name': e.get('name'),
            'date': e.get('dates', {}).get('start', {}).get('localDate'),
            'category': e.get('classifications', [{}])[0].get('segment', {}).get('name'),
            'subcategory': e.get('classifications', [{}])[0].get('genre', {}).get('name'),
            'venue': e.get('_embedded', {}).get('venues', [{}])[0].get('name'),
            'price_min': e.get('priceRanges', [{}])[0].get('min') if e.get('priceRanges') else None,
            'price_max': e.get('priceRanges', [{}])[0].get('max') if e.get('priceRanges') else None,
        })
    return events

print('Funciones definidas OK')

Funciones definidas OK


In [6]:
# Descarga paginada: dic 2025 → dic 2026
START_DATE = '2025-12-01'
END_DATE   = '2026-12-31'

all_events = []
page = 0

# Primera petición para saber el total de páginas
first_response = fetch_events_page(page, START_DATE, END_DATE)
page_info = first_response.get('page', {})
total_pages = page_info.get('totalPages', 1)
total_elements = page_info.get('totalElements', 0)

print(f'Total eventos encontrados: {total_elements}')
print(f'Total páginas: {total_pages}')
print('Descargando...')

all_events.extend(parse_events(first_response))

# Ticketmaster limita a 5 páginas en plan gratuito
max_pages = min(total_pages, 5)

for p in range(1, max_pages):
    time.sleep(0.3)  # respetar rate limit
    response_data = fetch_events_page(p, START_DATE, END_DATE)
    all_events.extend(parse_events(response_data))
    print(f'  Página {p+1}/{max_pages} — eventos acumulados: {len(all_events)}')

print(f'Descarga completada. Total eventos: {len(all_events)}')

Total eventos encontrados: 1310
Total páginas: 7
Descargando...
  Página 2/5 — eventos acumulados: 400
  Página 3/5 — eventos acumulados: 600
  Página 4/5 — eventos acumulados: 800
  Página 5/5 — eventos acumulados: 1000
Descarga completada. Total eventos: 1000


In [7]:
# Convertir a DataFrame y guardar
if not all_events:
    print('No se encontraron eventos. Revisa las fechas o la API key.')
else:
    df_eventos = pd.DataFrame(all_events)
    df_eventos['date'] = pd.to_datetime(df_eventos['date'])
    df_eventos.drop_duplicates(subset='id', inplace=True)

    output_path = RAW_DIR / 'eventos_barcelona.csv'
    df_eventos.to_csv(output_path, index=False)

    print(f'Guardado en: {output_path}')
    print(f'Shape: {df_eventos.shape}')
    print(f'Categorías encontradas: {df_eventos["category"].value_counts().to_dict()}')
    display(df_eventos.head(10))

Guardado en: ..\data\raw\eventos_barcelona.csv
Shape: (1000, 8)
Categorías encontradas: {'Miscellaneous': 968, 'Music': 31, 'Arts & Theatre': 1}


,id,name,date,category,subcategory,venue,price_min,price_max
0,LvZ18Q_gT3GpfgYZvOS32,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
1,LvZ18Q_gT3GpfgYZvOS38,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
2,LvZ18Q_gT3GpfgYZvOS3o,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
3,LvZ18Q_gT3GpfgYZvOS39,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
4,LvZ18Q_gT3GpfgYZvOS3b,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
5,LvZ18Q_gT3GpfgYZvOS3_,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
6,LvZ18Q_gT3GpfgYZvOS3Q,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
7,LvZ18Q_gT3GpfgYZvOS3h,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
8,LvZ18Q_gT3GpfgYZvOS3j,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None
9,LvZ18Q_gT3GpfgYZvOS3q,Antarctica Experience,2026-04-22,Miscellaneous,Family,Cúpula de las Arenas,None,None


---
## 3. Resumen de datos disponibles

In [8]:
print('=' * 50)
print('RESUMEN DE DATOS EN data/raw/')
print('=' * 50)

for f in sorted(RAW_DIR.glob('*.csv')):
    df_temp = pd.read_csv(f, nrows=1)  # solo leer cabecera para ver columnas
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f'\n{f.name}')
    print(f'  Tamaño : {size_mb:.1f} MB')
    print(f'  Columnas: {list(df_temp.columns)}')

RESUMEN DE DATOS EN data/raw/

calendar.csv
  Tamaño : 230.2 MB
  Columnas: ['listing_id', 'date', 'available', 'price', 'adjusted_price', 'minimum_nights', 'maximum_nights']

clima_barcelona.csv
  Tamaño : 0.0 MB
  Columnas: ['date', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'weathercode', 'fuente']

eventos_barcelona.csv
  Tamaño : 0.1 MB
  Columnas: ['id', 'name', 'date', 'category', 'subcategory', 'venue', 'price_min', 'price_max']

listings.csv
  Tamaño : 36.9 MB
  Columnas: ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'h